In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import random

# --- 1. Dataset simulato ---
class DialogueDataset(Dataset):
    def __init__(self, num_samples=100, max_dialogue_len=6, tokenizer_name="roberta-base", max_len=32):
        self.num_samples = num_samples
        self.max_dialogue_len = max_dialogue_len
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_len = max_len
        self.data = []
        self._simulate_data()
    
    def _simulate_data(self):
        messages_pool = [
            "Ciao!", "Oh davvero?", "Hai finito il lavoro?", "Sì, finalmente!", 
            "Che bello!", "Non ci credo...", "Davvero interessante", "Ma dai!"
        ]
        for _ in range(self.num_samples):
            dialogue_len = random.randint(3, self.max_dialogue_len)
            dialogue = [random.choice(messages_pool) for _ in range(dialogue_len)]
            speakers = [i % 2 for i in range(dialogue_len)]  # alternanza U1/U2
            label = random.randint(0, 1)  # 0 = non appropriato, 1 = appropriato
            self.data.append({
                "dialogue": dialogue,
                "speakers": speakers,
                "label": label
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        input_ids = []
        attention_mask = []
        for msg in item["dialogue"]:
            encoded = self.tokenizer.encode_plus(
                msg, max_length=self.max_len,
                padding='max_length', truncation=True,
                return_tensors='pt'
            )
            input_ids.append(encoded['input_ids'])
            attention_mask.append(encoded['attention_mask'])
        
        input_ids = torch.cat(input_ids, dim=0)      # [N, L]
        attention_mask = torch.cat(attention_mask, dim=0)  # [N, L]
        speaker_ids = torch.tensor(item["speakers"])       # [N]
        label = torch.tensor(item["label"], dtype=torch.float)  # scalar
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "speaker_ids": speaker_ids,
            "label": label
        }

# --- 2. Modello EMOGEN-lite per Irony Appropriateness ---
class IronyAppropriatenessModel(nn.Module):
    def __init__(self, transformer_model="roberta-base", hidden_dim=1024):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(transformer_model)
        self.hidden_dim = hidden_dim
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8)
        self.conversation_block = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.intra_speaker_block = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.inter_speaker_block = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, input_ids, attention_mask, speaker_ids):
        B, N, L = input_ids.shape
        
        # Encode utterances
        x = input_ids.view(B*N, L)
        mask = attention_mask.view(B*N, L)
        r = self.encoder(input_ids=x, attention_mask=mask).last_hidden_state[:,0,:]  # [CLS]
        r = r.view(B, N, self.hidden_dim)
        
        # Masks
        speaker_matrix = speaker_ids.unsqueeze(1) == speaker_ids.unsqueeze(2)
        intra_mask = speaker_matrix  # True = attend
        inter_mask = ~speaker_matrix
        
        # Transformer blocks
        A_c = self.conversation_block(r.transpose(0,1)).transpose(0,1)
        A_ra = self.intra_speaker_block(r.transpose(0,1)).transpose(0,1)
        A_er = self.inter_speaker_block(r.transpose(0,1)).transpose(0,1)
        
        # Self-Attention Pooling
        O = torch.stack([A_c, A_ra, A_er], dim=2)  # [B, N, 3, H]
        K = self.key_proj(O)
        Q = self.query_proj(O)
        V = self.value_proj(O)
        attn_scores = torch.softmax(torch.matmul(Q, K.transpose(-2,-1))/self.hidden_dim**0.5, dim=-1)
        T = torch.matmul(attn_scores, V)
        T_hat = torch.max(T, dim=2).values  # [B,N,H]
        
        # Ultimo messaggio U1
        last_u1_idx = (speaker_ids == 0).sum(dim=1) - 1
        batch_indices = torch.arange(B)
        u1_embedding = T_hat[batch_indices, last_u1_idx, :]
        
        out = self.classifier(u1_embedding)
        return out

# --- 3. Setup dataset e dataloader ---
dataset = DialogueDataset(num_samples=50)  # piccolo dataset simulato
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = IronyAppropriatenessModel().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCELoss()

# --- 4. Training loop ---
num_epochs = 2  # per test, aumentare per dataset reale
for epoch in range(num_epochs):
    total_loss = 0
    model.train()
    
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        speaker_ids = batch['speaker_ids'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        output = model(input_ids.unsqueeze(0), attention_mask.unsqueeze(0), speaker_ids.unsqueeze(0))
        loss = criterion(output.squeeze(), labels.float())
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

# --- 5. Test rapido sul primo esempio ---
model.eval()
with torch.no_grad():
    example = dataset[0]
    input_ids = example['input_ids'].unsqueeze(0).to(device)
    attention_mask = example['attention_mask'].unsqueeze(0).to(device)
    speaker_ids = example['speaker_ids'].unsqueeze(0).to(device)
    pred = model(input_ids, attention_mask, speaker_ids)
    print("Probabilità ironia appropriata:", pred.item())


c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\.venv\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


RuntimeError: stack expects each tensor to be equal size, but got [4, 32] at entry 0 and [5, 32] at entry 1